# TRELLIS.2 — RunPod GPU Setup & Inference

Tested on **A100 / H100 (24 GB+ VRAM)** with CUDA 12.4.  
Run cells top-to-bottom on a fresh RunPod PyTorch pod.

## 1. Verify GPU

In [1]:
!nvidia-smi

Mon Apr  6 17:38:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           On  |   00000000:86:00.0 Off |                    0 |
| N/A   33C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Mar_28_02:18:24_PDT_2024
Cuda compilation tools, release 12.4, V12.4.131
Build cuda_12.4.r12.4/compiler.34097967_0


## 2. Clone Repo (skip if already present)

In [3]:
import os

REPO_DIR = "/workspace/TRELLIS.2"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RahulBhalleySP/TRELLIS.2.git {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

# Persist repo path so bash cells can find it without hardcoding
with open('/tmp/trellis2_repo', 'w') as f:
    f.write(REPO_DIR)

Repo already present at /workspace/TRELLIS.2
Working directory: /workspace/TRELLIS.2


## 3. Install Dependencies

All packages — PyTorch, Python libraries, CUDA extensions — are built/downloaded once and cached to `/workspace/wheels/`.  
On subsequent runs the cell just restores from cache and exits in seconds.

### 3a. PyTorch (CUDA 12.4)

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q

### 3b. Basic Python packages

In [ ]:
!pip install "typing_extensions>=4.10.0" -q
!pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja \
             trimesh transformers gradio==6.0.1 tensorboard pandas lpips zstandard \
             kornia timm -q
!pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8 -q

# pillow-simd is a drop-in faster Pillow; fall back to stock Pillow if build fails
!apt-get update -qq && apt-get install -y libjpeg-turbo8-dev zlib1g-dev -qq 2>/dev/null || \
 apt-get install -y libjpeg-dev zlib1g-dev -qq 2>/dev/null || true
!pip install pillow-simd -q 2>/dev/null || pip install Pillow -q
!python -c "from PIL import Image; print('Pillow OK:', Image.__version__)"

### 3c. Flash-Attention

In [ ]:
!pip install flash-attn==2.7.3 -q

### 3d. CUDA extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, O-Voxel)

Wheels are compiled once and cached to `/workspace/wheels/`.  
On subsequent runs the cached `.whl` files are installed directly — no recompilation.

In [ ]:
%%bash
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore
set -eo pipefail

WHEEL_DIR="/workspace/wheels"
WHEEL_ZIP="/workspace/wheels.zip"
EXT_DIR="/tmp/extensions"

# ── Restore cache from zip if folder not yet present ─────────────────────────
if [ ! -d "$WHEEL_DIR" ] && [ -f "$WHEEL_ZIP" ]; then
    echo "Restoring from $WHEEL_ZIP ..."
    python3 -m zipfile -e "$WHEEL_ZIP" /workspace
fi

# ── Cache hit: install and exit ───────────────────────────────────────────────
if [ -d "$WHEEL_DIR" ] && ls "$WHEEL_DIR"/*.whl &>/dev/null; then
    echo "Cache hit — installing from $WHEEL_DIR"
    pip install "$WHEEL_DIR"/*.whl --no-deps -q
    echo "All CUDA extensions installed from cache."
    exit 0
fi

# ── Cache miss: build from source (~10-20 min) ────────────────────────────────
echo "No cache found — building CUDA extensions from source ..."
mkdir -p "$WHEEL_DIR" "$EXT_DIR"

build() {
    local name="$1" url="$2" branch="$3"
    echo ""
    echo "==> $name"
    local dir="$EXT_DIR/$name"
    if [ ! -d "$dir" ]; then
        local args=(-q --recursive)
        [ -n "$branch" ] && args+=(-b "$branch")
        git clone "${args[@]}" "$url" "$dir"
    fi
    pip wheel "$dir" --no-build-isolation --no-deps -w "$WHEEL_DIR" -q
    echo "    OK"
}

# URLs and branches match setup.sh exactly
build nvdiffrast "https://github.com/NVlabs/nvdiffrast.git"     "v0.4.0"
build nvdiffrec  "https://github.com/JeffreyXiang/nvdiffrec.git" "renderutils"
build cumesh     "https://github.com/JeffreyXiang/CuMesh.git"    ""
build flex_gemm  "https://github.com/JeffreyXiang/FlexGEMM.git"  ""

# o-voxel: local source, needs Eigen submodule
OVOXEL_SRC=$(find /workspace -maxdepth 4 -name "o-voxel" -type d \
             -not -path '*/.git/*' 2>/dev/null | head -1)
[ -z "$OVOXEL_SRC" ] && { echo "[error] o-voxel not found under /workspace" >&2; exit 1; }
if [ ! -f "$OVOXEL_SRC/third_party/eigen/Eigen/Dense" ]; then
    echo "Initializing Eigen submodule ..."
    git -C "$(dirname "$OVOXEL_SRC")" submodule update --init --recursive \
        -- o-voxel/third_party/eigen -q \
        || git -C "$OVOXEL_SRC" submodule update --init --recursive -q
fi
rm -rf "$EXT_DIR/o-voxel"
cp -r "$OVOXEL_SRC" "$EXT_DIR/o-voxel"
build o_voxel "" ""

# ── Install all built wheels ──────────────────────────────────────────────────
echo ""
echo "Installing wheels ..."
pip install "$WHEEL_DIR"/*.whl --no-deps -q

# ── Save zip for future pods ──────────────────────────────────────────────────
echo "Saving $WHEEL_ZIP ..."
python3 - <<'PYEOF'
import zipfile, pathlib
wd = pathlib.Path("/workspace/wheels")
zp = pathlib.Path("/workspace/wheels.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(wd.rglob("*")):
        zf.write(f, f.relative_to(wd.parent))
print(f"Saved {zp} ({zp.stat().st_size/1e6:.1f} MB)")
PYEOF

echo "Done."

In [ ]:
import importlib, typing_extensions
importlib.reload(typing_extensions)
print("typing_extensions reloaded:", typing_extensions.__version__)

## 4. Verify Imports

In [11]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla V100-SXM2-32GB
VRAM: 34.1 GB


In [12]:
import sys
sys.path.insert(0, '/workspace/TRELLIS.2')

import cv2
import imageio
from PIL import Image
import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

print("All imports OK")

[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn
All imports OK


## 5. HuggingFace Login

Required to download the TRELLIS.2-4B weights. Paste your token below (read-access is enough).

In [ ]:
from huggingface_hub import login, whoami

login("your_huggingface_token_here")  # replace with your token

info = whoami()
print(f"Logged in as: {info['name']} ({info.get('email', 'no email on record')})")

## 6. Load Pipeline

Downloads ~4B-parameter model weights from HuggingFace on first run (~15 GB).

In [13]:
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()
print("Pipeline ready.")

pipeline.json: 0.00B [00:00, ?B/s]

ss_dec_conv3d_16l8_fp16.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

ckpts/ss_dec_conv3d_16l8_fp16.safetensor(…):   0%|          | 0.00/148M [00:00<?, ?B/s]

ss_flow_img_dit_1_3B_64_bf16.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

ckpts/ss_flow_img_dit_1_3B_64_bf16.safet(…):   0%|          | 0.00/2.58G [00:00<?, ?B/s]

...

ckpts/tex_dec_next_dc_f16c32_fp16.safete(…):   0%|          | 0.00/948M [00:00<?, ?B/s]

(…)flow_imgshape2tex_dit_1_3B_512_bf16.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 2584.67 MB. The target location /root/.cache/huggingface/hub/models--microsoft--TRELLIS.2-4B/blobs only has 2035.10 MB free disk space.
  warnings.warn(


ckpts/slat_flow_imgshape2tex_dit_1_3B_51(…):   0%|          | 0.00/2.58G [00:00<?, ?B/s]

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69d3f20d-351e32820d53385435849fa9;db30c917-09e1-437e-b6df-943e63a0c614)

Repository Not Found for url: https://huggingface.co/ckpts/slat_flow_imgshape2tex_dit_1_3B_512_bf16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

## 7. Load Environment Map (for PBR rendering)

In [ ]:
envmap = EnvMap(
    torch.tensor(
        cv2.cvtColor(
            cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED),
            cv2.COLOR_BGR2RGB
        ),
        dtype=torch.float32,
        device='cuda'
    )
)
print("Environment map loaded.")

## 8. Pick Input Images

Any PNG / WebP with a clean background works.  
Change `image_paths` to your own files or upload them to `/workspace/`.

In [ ]:
import glob
from IPython.display import display

# Use a handful of the bundled example images
all_examples = sorted(glob.glob('assets/example_image/*.webp'))[:4]
all_examples += ['assets/example_image/T.png']  # also include the PNG

print("Images to process:")
for p in all_examples:
    img = Image.open(p).convert('RGB')
    img.thumbnail((128, 128))
    display(img)
    print(p)

## 9. Run Inference

Default pipeline type is `'1024_cascade'` (best quality).  
Use `'512'` for a quick test (~30 s vs ~3 min on A100).

In [ ]:
import time

PIPELINE_TYPE = '1024_cascade'  # '512' for fast test
OUTPUT_DIR = '/workspace/trellis2_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for img_path in all_examples:
    name = os.path.splitext(os.path.basename(img_path))[0]
    print(f"\n{'='*60}")
    print(f"Processing: {img_path}")

    image = Image.open(img_path)

    t0 = time.time()
    mesh = pipeline.run(image, pipeline_type=PIPELINE_TYPE)[0]
    mesh.simplify(16_777_216)  # nvdiffrast vertex limit
    print(f"Inference: {time.time()-t0:.1f}s | "
          f"vertices={mesh.vertices.shape[0]:,} faces={mesh.faces.shape[0]:,}")

    # --- Render turntable video ---
    t1 = time.time()
    frames = render_utils.make_pbr_vis_frames(
        render_utils.render_video(mesh, envmap=envmap)
    )
    video_path = os.path.join(OUTPUT_DIR, f'{name}.mp4')
    imageio.mimsave(video_path, frames, fps=15)
    print(f"Video saved: {video_path}  ({time.time()-t1:.1f}s)")

    # --- Export GLB ---
    t2 = time.time()
    glb = o_voxel.postprocess.to_glb(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        use_tqdm          = True,
    )
    glb_path = os.path.join(OUTPUT_DIR, f'{name}.glb')
    glb.export(glb_path, extension_webp=True)
    print(f"GLB saved:   {glb_path}  ({time.time()-t2:.1f}s)")

    # --- Export USDZ ---
    t3 = time.time()
    usdz_path = os.path.join(OUTPUT_DIR, f'{name}.usdz')
    o_voxel.postprocess.to_usdz(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        file_path         = usdz_path,
        use_tqdm          = True,
    )
    print(f"USDZ saved:  {usdz_path}  ({time.time()-t3:.1f}s)")

    torch.cuda.empty_cache()

print("\nAll done.")

## 10. Preview Outputs

In [ ]:
# List generated files
!ls -lh {OUTPUT_DIR}

In [ ]:
# Preview the first video inline
from IPython.display import Video
import glob

videos = sorted(glob.glob(f'{OUTPUT_DIR}/*.mp4'))
if videos:
    display(Video(videos[0], embed=True, width=512))

## 11. Download Outputs

Zip everything up for easy download from the RunPod file manager.

In [ ]:
!zip -r /workspace/trellis2_outputs.zip {OUTPUT_DIR}
print("Ready: /workspace/trellis2_outputs.zip")

---
## Optional: Single-Image Quick Test

Drop your own image at `/workspace/my_image.png` and run this cell.

In [ ]:
MY_IMAGE = '/workspace/my_image.png'  # <-- change this

if os.path.exists(MY_IMAGE):
    image = Image.open(MY_IMAGE)
    mesh = pipeline.run(image, pipeline_type='512')[0]  # '512' for speed
    mesh.simplify(16_777_216)

    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces,
        attr_volume=mesh.attrs, coords=mesh.coords,
        attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
        aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=500_000, texture_size=2048,
        remesh=True, remesh_band=1, remesh_project=0,
    )
    out = '/workspace/my_output.glb'
    glb.export(out, extension_webp=True)
    print(f"Saved: {out}")

    torch.cuda.empty_cache()
else:
    print(f"File not found: {MY_IMAGE}")